# 01 — Validación del ETL

El procesamiento ETL se ejecuta via scripts:
```bash
python scripts/prepare_mic.py        # mic fijos → predictions_geo.parquet
python scripts/prepare_mobile.py     # móvil → predictions_mobile.parquet
```
Este notebook valida la calidad del resultado.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

_dp = Path('../data/processed')
for _f in ['predictions_geo.parquet', 'tracks.parquet']:
    if not (_dp / _f).exists():
        raise FileNotFoundError(
            f"{_f} no encontrado.\nEjecutar primero:\n"
            "  python scripts/prepare_mic.py\n"
            "  python scripts/prepare_mobile.py  (opcional, para datos móvil)"
        )

pred_geo = pd.read_parquet(_dp / 'predictions_geo.parquet')
tracks   = pd.read_parquet(_dp / 'tracks.parquet')

CLASS_NAMES = {0:"Horn",1:"Siren",2:"Pets",3:"Physiological",
               4:"Speech",5:"Ring Tone",6:"Vibrating",7:"Notifications",8:"Cry"}
pred_geo['class_name'] = pred_geo['class'].map(CLASS_NAMES)

print(f"Eventos geolocalizados : {len(pred_geo)}")
print(f"Trackpoints GPS        : {len(tracks)}")
print(f"Fuentes                : {pred_geo['source'].value_counts().to_dict()}")
print(f"Días                   : {sorted(pred_geo['date'].unique())}")

## 1. Cobertura de Datos por Fuente
¿Qué días hay datos, de qué fuente y cuántos eventos?

In [ ]:
coverage = (
    pred_geo.groupby(['date', 'source'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
coverage['total'] = coverage.drop(columns='date').sum(axis=1)
coverage = coverage.sort_values('date')
display(coverage)

ax = coverage.set_index('date')[coverage.columns[1:-1]].plot(
    kind='bar', stacked=True, figsize=(12, 4),
    color=['steelblue', 'coral'][:len(coverage.columns)-2],
    title='Eventos por día y fuente'
)
ax.set_xlabel('Día'); ax.set_ylabel('Nº eventos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Validación Temporal — Overlap GPS ↔ Predicciones
Comprobamos que los rangos horarios de predicciones y GPS coinciden por día y fuente.
Un overlap_min ≤ 0 indica desincronía: las predicciones no caen dentro de la ventana GPS.

In [ ]:
results = []
for (date, source), grp in pred_geo.groupby(['date', 'source']):
    trk_day = tracks[tracks['date'] == date]
    if trk_day.empty:
        continue
    gps_min, gps_max   = trk_day['time'].min(), trk_day['time'].max()
    pred_min, pred_max = grp['t_start'].min(), grp['t_start'].max()
    overlap = (min(gps_max, pred_max) - max(gps_min, pred_min)).total_seconds() / 60
    results.append({
        'date': date, 'source': source,
        'gps_start':  gps_min.strftime('%H:%M'),
        'gps_end':    gps_max.strftime('%H:%M'),
        'pred_start': pred_min.strftime('%H:%M'),
        'pred_end':   pred_max.strftime('%H:%M'),
        'overlap_min': round(overlap, 1),
    })

df_overlap = pd.DataFrame(results).sort_values(['date','source'])
display(df_overlap.style.apply(
    lambda col: ['background-color: #ffcccc' if v <= 0 else '' for v in col]
    if col.name == 'overlap_min' else ['']*len(col), axis=0
))

bad = df_overlap[df_overlap['overlap_min'] <= 0]
if len(bad):
    print(f"\n⚠️  Sin solapamiento: {bad[['date','source']].values.tolist()}")
else:
    print("\n✅ Todas las predicciones tienen solapamiento temporal con GPS.")

## 3. Calidad del Join — % Geolocalizados por Día

In [ ]:
# Cargar predicciones antes del join para comparar totales
_pred_mic = pd.read_parquet(_dp / 'predictions_mic.parquet') if (_dp/'predictions_mic.parquet').exists() else pd.DataFrame()
_pred_mob_path = _dp / 'predictions_mobile.parquet'
_pred_mob = pd.read_parquet(_pred_mob_path) if _pred_mob_path.exists() else pd.DataFrame()

if not _pred_mob.empty:
    _pred_mob = _pred_mob.rename(columns={'timestamp_onset':'t_start','class_id':'class'})
    _pred_mob['t_start'] = pd.to_datetime(_pred_mob['t_start'], utc=True)
    _pred_mob['date'] = _pred_mob['t_start'].dt.strftime('%d-%m-%Y')

_all_preds = pd.concat([_pred_mic, _pred_mob], ignore_index=True) if not _pred_mob.empty else _pred_mic

quality = []
for date in sorted(_all_preds['date'].unique()):
    total   = len(_all_preds[_all_preds['date'] == date])
    joined  = len(pred_geo[pred_geo['date'] == date])
    quality.append({'date': date, 'total_preds': total,
                    'geolocalizados': joined,
                    'pct': f"{joined/total*100:.1f}%" if total else "0%"})

df_q = pd.DataFrame(quality).sort_values('date')
display(df_q)
print(f"\nGlobal: {len(pred_geo)} / {len(_all_preds)} = {len(pred_geo)/max(1,len(_all_preds)):.1%}")

print("\nDetecciones geolocalizadas por micrófono (solo mic):")
mic_data = pred_geo[pred_geo['source']=='mic']
if not mic_data.empty:
    print(mic_data['microfono_id'].value_counts().to_string())

## 4. Distribución de Clases por Fuente
Comparación mic vs móvil: ¿detectan las mismas clases con la misma frecuencia?

In [ ]:
sources = pred_geo['source'].unique()
if len(sources) > 1:
    fig, axes = plt.subplots(1, len(sources), figsize=(14, 5), sharey=False)
    for ax, src in zip(axes, sorted(sources)):
        grp = pred_geo[pred_geo['source'] == src]
        counts = grp['class_name'].value_counts()
        counts.plot(kind='bar', ax=ax, color='steelblue' if src == 'mic' else 'coral')
        ax.set_title(f'Fuente: {src} ({len(grp)} eventos)')
        ax.set_xlabel('Clase'); ax.set_ylabel('Nº eventos')
        ax.tick_params(axis='x', rotation=45)
    plt.suptitle('Distribución de clases por fuente', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    pred_geo['class_name'].value_counts().plot(
        kind='bar', figsize=(10, 4), color='steelblue',
        title=f'Distribución de clases (fuente: {sources[0]})'
    )
    plt.xlabel('Clase'); plt.ylabel('Nº eventos')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 5. Anomalías Conocidas

- **11-03-2026**: sin solapamiento temporal entre GPS y predicciones. Las predicciones (20:16–20:37 UTC) caen fuera de la ventana GPS (13:59–19:38 UTC). Causa probable: desfase en el timestamp del archivo de audio. Día excluido del análisis geoespacial.